# Editor de proyecto (anywidget)

Comprobación manual del widget: lo que no pueden verificar los tests de Python.

**Antes de empezar**: selecciona el kernel del `.venv` del proyecto (en VS Code,
arriba a la derecha → *Select Kernel* → *Python Environments* → `.venv`).

La interfaz es maestro-detalle: a la izquierda los componentes agrupados por tipo,
a la derecha el formulario del que tengas seleccionado. Los campos se generan del
esquema, así que cada uno sale con su unidad, sus límites y sus opciones.

La primera vez, el navegador descarga `ajv` de esm.sh: hace falta conexión.

## 1. Proyecto pequeño

In [10]:
import opensimula as osm

sim = osm.Simulation()
sim.console_print = False
pro = sim.new_project("proyecto")
pro.read_json("../test/test_project_1.json")

editor = pro.editor()
editor

### Qué mirar

1. Que aparezcan los dos paneles, no una caja vacía ni `Error displaying widget`.
2. Pincha un `Material`. El formulario debe traer `conductivity` como campo
   numérico con la unidad `W/(m·K)` al lado, y `use_resistance` como casilla.
3. Pon `conductivity` a `-1`: el campo se marca en rojo, con
   `must be >= 0` debajo y en la barra de estado. Si dijera *"must NOT have
   additional properties"*, el discriminador de Ajv no estaría actuando.
4. Mira un `Building_surface`: `construction` tiene que ser un **desplegable con
   los nombres de las Construction del proyecto**, no un campo de texto libre.
5. Déjalo válido antes de seguir.

## 2. ¿Llegan los cambios al kernel?

In [11]:
# Cambia algo en el widget de arriba, espera medio segundo y ejecuta esto.
editor.value["components"][0]

{'name': 'comp 1',
 'description': 'Dummy component for testing',
 'type': 'Test_component',
 'boolean': True,
 'string': 'Hola mundo',
 'int': 24,
 'float': 34.5,
 'options': 'Two',
 'component': 'not_defined',
 'variable': 'not_defined = component.variable',
 'math_exp': '0.0',
 'boolean_list': [True, True],
 'string_list': ['Hola 1', 'Hola 2'],
 'int_list': [1, 2],
 'float_list': [1.1, 2.1],
 'options_list': ['Two', 'Two'],
 'component_list': ['not_defined', 'not_defined'],
 'variable_list': ['not_defined = component.variable'],
 'math_exp_list': ['0.0']}

In [12]:
print("válido:", editor.is_valid())
for linea in editor.error_report():
    print(" ", linea)

válido: True


Si `editor.value` no refleja lo que escribiste, la sincronización JS → Python
no funciona (consola del navegador: *Developer: Toggle Developer Tools*).

## 3. ¿Llegan los cambios del kernel al widget?

In [13]:
# Reasignar, nunca mutar: traitlets detecta los cambios por identidad.
editor.value = {**editor.value, "description": "cambiado desde Python"}

El panel de proyecto debe mostrar el nuevo `description` sin perder lo demás.

### 4. Para admitir los cambios hechos con el editor
editor.apply()

In [18]:
errors = editor.apply()
for m in errors:
    print(m.text if hasattr(m, "text") else m)
pro

,key,type,value,unit
0,name,Parameter_string,Proyecto 1,
1,description,Parameter_string,cambiado desde Python,
2,time_step,Parameter_int,3600,s
3,n_time_steps,Parameter_int,8760,
4,initial_time,Parameter_string,01/01/2001 00:00:00,
5,daylight_saving,Parameter_boolean,False,
6,daylight_saving_start_time,Parameter_string,25/03/2001 02:00:00,
7,daylight_saving_end_time,Parameter_string,28/10/2001 02:00:00,
8,n_max_iteration,Parameter_int,1000,
9,simulation_order,Parameter_string_list,"[Space_type, Building_surface, Solar_surface, Opening, Space, Building, HVAC_MZW_system, HVAC_perfect_system, HVAC_DX_system, HVAC_SZW_system, HVAC_water_system, Calculator]",


## 4. Un edificio real

In [14]:
hulc = sim.new_project("hulc")
hulc.read_json("edificio_curso_hulc.json")
print("componentes:", len(hulc.component_list()))

editor_hulc = hulc.editor()
editor_hulc

componentes: 150


Aquí se ve si aguanta un edificio de verdad: 150 componentes en 15 grupos.

**Prueba el aviso de referencias colgadas**: renombra una `Construction` y mira la
barra de estado. Debe avisar en amarillo de los `Building_surface` que se quedan
apuntando a un nombre que ya no existe. Eso no lo detecta el esquema — que un
nombre exista es propiedad del documento, no del tipo — así que es una
comprobación aparte.

## 5. Marimo

El mismo widget, envuelto:

```python
import marimo as mo
import opensimula as osm

sim = osm.Simulation()
pro = sim.new_project("proyecto")
pro.read_json("test/test_project_1.json")

editor = mo.ui.anywidget(pro.editor())
editor
```

En otra celda, `editor.value["value"]` es reactivo.

## Desarrollo del JS

Para tocar `editor.js` sin reiniciar el kernel, arranca con `ANYWIDGET_HMR=1`.

## 6. Llevar los cambios al proyecto

Editar el widget **no** modifica el proyecto: sólo cambia el documento en
`editor.value`. Para aplicarlo, `apply()`.

Antes de ejecutar la celda, sal del campo que estuvieras editando (clic fuera o
Enter): el formulario confirma al perder el foco y espera 400 ms más antes de
mandar el documento al kernel.

In [ ]:
errores = editor.apply()

for e in errores:
    # Si el documento no cumple el esquema, apply() no toca el proyecto y
    # devuelve los errores como diccionarios. Si es válido, reconstruye el
    # proyecto y devuelve los mensajes de check().
    print(e["message"] if isinstance(e, dict) else e.text)

`apply()` **reconstruye** el proyecto, no lo parchea. Dos consecuencias:

- Cualquier variable que apuntara a un componente de antes queda obsoleta; hay
  que volver a pedirlo con `pro.component(nombre)`.
- Los resultados de simulación se van con los componentes viejos: después de
  `apply()` hay que volver a `simulate()`.

Es lo que permite que renombrar funcione, porque los componentes se referencian
por nombre y las referencias se resuelven al cargar.

In [ ]:
# Sin navegador, is_valid() y error_report() siguen funcionando:
# se calculan en Python, no dependen de lo que reporte el widget.
print("válido:", editor.is_valid())
for linea in editor.error_report():
    print(" ", linea)